# Courbes d'entraînement et inspection des prédictions

Ce notebook lit `history.jsonl`, `metrics.json` et `predictions.jsonl` produits par le pipeline. Il compare la validation au test officiel et affiche quelques erreurs représentatives.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein
from PIL import Image

def locate_run():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        run = root / 'runs' / 'baseline'
        if (run / 'history.jsonl').is_file(): return run
        run = root / 'outputs' / 'arabic_ocr' / 'runs' / 'baseline'
        if (run / 'history.jsonl').is_file(): return run
    raise FileNotFoundError('Run baseline introuvable.')

RUN = locate_run()
PROJECT = RUN.parents[2] if RUN.parent.name == 'arabic_ocr' else RUN.parents[3]
print('Run :', RUN)

In [ ]:
history = [json.loads(line) for line in (RUN / 'history.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
epochs = pd.DataFrame([row for row in history if row.get('epoch', 0) > 0])
val = pd.json_normalize(epochs['validation'])
curves = pd.concat([epochs[['epoch','train_ctc_loss']].reset_index(drop=True), val.add_prefix('val_')], axis=1)
curves[['epoch','train_ctc_loss','val_cer','val_wer','val_exact_accuracy']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(curves['epoch'], 100*curves['val_cer'], 'o-', label='CER validation')
axes[0].plot(curves['epoch'], 100*curves['val_wer'], 's-', label='WER validation')
axes[0].set(xlabel='époque', ylabel='erreur (%)', title='Validation')
axes[0].legend()
axes[1].plot(curves['epoch'], curves['train_ctc_loss'], 'o-', color='#23724c', label='perte CTC')
axes[1].plot(curves['epoch'], 1-curves['val_exact_accuracy'], 's--', color='#c4552b', label='1 - exactitude')
axes[1].set(xlabel='époque', title='Apprentissage')
axes[1].legend()
for axis in axes: axis.grid(alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
metrics_path = RUN / 'test' / 'metrics.json'
pred_path = RUN / 'test' / 'predictions.jsonl'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
predictions = pd.DataFrame([json.loads(line) for line in pred_path.read_text(encoding='utf-8').splitlines() if line.strip()])
print({key: metrics[key] for key in ('samples','cer','wer','exact_accuracy','seconds','lines_per_second')})
predictions['line_cer'] = predictions.apply(lambda row: Levenshtein.distance(row['reference'], row['prediction']) / max(1, len(row['reference'])), axis=1)
predictions.sort_values('line_cer', ascending=False).head(10)[['id','line_cer','reference','prediction']]

In [ ]:
data_dir = PROJECT / 'work' / 'data' / 'gazette'
images = {json.loads(line)['id']: json.loads(line)['image'] for line in (data_dir/'test.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()}
worst = predictions.sort_values('line_cer', ascending=False).head(6)
fig, axes = plt.subplots(len(worst), 1, figsize=(16, max(4, 2.2*len(worst))))
axes = [axes] if len(worst) == 1 else axes
for axis, (_, row) in zip(axes, worst.iterrows()):
    with Image.open(images[row['id']]) as image: axis.imshow(image.convert('L'), cmap='gray')
    axis.set_title(f"CER={row['line_cer']:.1%} | ref: {row['reference'][:140]} | pred: {row['prediction'][:140]}", loc='left', fontsize=8)
    axis.axis('off')
plt.tight_layout()
plt.show()